# 검색을 더 똑똑하게 만들기

**위니브 카페 CS 챗봇 만들기**


## 배울 것 

> 챗봇이 가끔 **엉뚱한 청크**를 가져와 헛소리를 합니다.
> 그 검색을 더 똑똑하게 만드는 3가지 처방을 배웁니다.

| 단계 | 내용 |
|------|------|
| 1️⃣ 이론 | RAG가 틀리면 왜 '검색'을 의심하나? 3가지 처방 |
| 2️⃣ 기본 | 처방 3개의 사용법 (필터·MMR·질문확장) |
| 3️⃣ 응용 | 처방을 조합하고 전후 비교하기 |
| 4️⃣ 실습 | 위니브에서 검색 실패 재현 → 처방으로 고치기 |


# 1️⃣ 단계 — 이론 

## 완성했는데, 가끔 헛소리를 한다

RAG 챗봇을 완성했습니다. 대부분 잘 답하지만,
가끔 이런 일이 생깁니다.

> 🧑 고객: "**원두** 환불 규정 알려주세요"
> 🤖 챗봇: "**디저트**는 신선식품이라 단순변심 반품 불가합니다."
> 🧑 고객: "...원두를 물어봤는데요?"

답이 **틀린 건 아닙니다.** 디저트 규정은 실제로 그렇습니다.
문제는 **고객이 물어본 게 아니라는 것.** 그냥 **검색이 엉뚱한
청크를 물어와서** AI가 그걸 성실히 답한 거죠.

## RAG가 틀리면, 범인은 보통 '검색'

챗봇이 이상하면 보통 "AI가 멍청한가?" "프롬프트가 문제인가?"
의심합니다. 

```
질문 → [② 검색] → [③ 정리] → [④ 프롬프트] → [⑤ AI] → 답
```

⑤ AI는 **②검색이 가져다준 청크 안에서만** 답합니다.
②가 엉뚱한 청크를 주면, 아무리 똑똑한 AI도 좋은 답을 못 합니다.

> 🧠 **진단 원칙**: RAG가 틀리면 *제일 먼저 검색 결과를 본다.*
> "AI에게 준 청크가 애초에 맞았나?" 가 1번입니다.
>
> 💡 비유: 요리가 맛없으면 요리사보다 **상한 재료**를 먼저
> 의심하는 것과 같습니다.

## 검색 실패 3가지 모습과 처방

| 증상 | 위니브 예시 | 처방 |
|------|-------------|------|
| ① 엉뚱한 분류가 섞임 | 원두 질문에 디저트 청크 | **필터** |
| ② 비슷한 청크만 잔뜩 | 똑같은 환불 문장 3개 | **MMR** |
| ③ 질문이 너무 짧고 모호 | "그거 언제 와요?" | **질문 확장** |

**처방① 필터** — 분류 꼬리표(`category`)로 검색
범위를 미리 좁힘. "환불 질문이면 환불 청크만 보기."
> 💡 비유: "파이썬 요리"를 찾는데 "뱀(파이썬) 사육서"가
> 나오면, "요리 코너에서만 찾아 줘"라고 조건 거는 것.

**처방② MMR** — "관련 있되 *서로 안 겹치게*" 청크를 골라줌.
> 💡 비유: 팀에 잘하는 사람만 3명 뽑았더니 다 디자이너 →
> 개발자가 없어 프로젝트가 안 굴러감. 관련성 + **다양성**이
> 같이 필요합니다.

**처방③ 질문 확장** — 짧은 질문을 AI가 여러 표현으로 바꿔 쓴
뒤 각각 검색해 합침.
> 💡 비유: 구글에 "환불"만 치는 사람보다 "환불 신청 후 입금
> 며칠"이라 치는 사람이 잘 찾죠. 질문을 풍부하게 만드는 것.
> (AI를 더 써서 느리고 비쌈 — 트레이드오프)

> 📌 핵심: 처방은 외우는 게 아니라 **증상 보고 고르는 것.**
> 그리고 **챗봇은 그대로 두고, 검색 부품만 갈아 끼웁니다.**

---
👉 아래부터 코드입니다. 한 줄씩 천천히 갑니다.


# 2️⃣ 단계 — 처방 3개 사용법 익히기

작은 예제로 3가지 처방을 하나씩 써봅니다.

### 셀 1 — 도구 설치

In [2]:
# 버전을 명시적으로 고정합니다 (이 조합이 서로 호환됨 — 에러 방지 핵심)
# langchain 1.x 부터 MultiQueryRetriever 가 langchain-classic 으로 이동했습니다.
!pip install -q -U \
    "langchain>=1.0" \
    "langchain-classic" \
    "langchain-openai" \
    "langchain-community" \
    "langchain-chroma" \
    "langchain-text-splitters" \
    "langchain-huggingface" \
    "python-dotenv" \
    "sentence-transformers"

### 셀 2 — 키 + 임베딩 + 연습 데이터

키 읽고, 임베딩 준비하고, 일부러 **비슷한 환불 문장 + 다른
분류**를 섞은 연습용 벡터 DB를 만듭니다.

In [3]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

has_openai = bool(os.environ.get("OPENAI_API_KEY"))

if has_openai:
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("✅ OpenAI 키 OK (질문확장까지 가능)")
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    print("⚠️ 키 없음 — 필터·MMR은 OK, 질문확장만 건너뜁니다")

from langchain_core.documents import Document
from langchain_chroma import Chroma

# 일부러 비슷한 환불 문장 4개 + 다른 분류 2개
sample_docs = [
    Document(page_content="환불은 수령 후 7일 이내 신청 가능합니다.",       metadata={"category": "refund"}),
    Document(page_content="단순 변심 환불은 7일 안에만 받습니다.",          metadata={"category": "refund"}),
    Document(page_content="환불 신청은 수령일 기준 7일 이내여야 합니다.",     metadata={"category": "refund"}),
    Document(page_content="환불 처리는 반품 확인 후 영업일 3일 이내입니다.",  metadata={"category": "refund"}),
    Document(page_content="배송비는 3,000원, 5만원 이상 무료입니다.",        metadata={"category": "shipping"}),
    Document(page_content="회원 등급은 누적 구매액으로 결정됩니다.",         metadata={"category": "membership"}),
]
db = Chroma.from_documents(sample_docs, embeddings)
print("연습용 벡터 DB 준비 완료 ✅ (환불 4개 + 배송/멤버십)")

✅ OpenAI 키 OK (질문확장까지 가능)
연습용 벡터 DB 준비 완료 ✅ (환불 4개 + 배송/멤버십)


**이 셀이 한 일**: 키·임베딩을 준비하고, 처방 효과를 보기
좋게 **일부러 비슷한 문장이 몰린** 연습 데이터를 만들었습니다.

### 셀 3 — 처방① 필터: 분류로 범위 좁히기

3교시에서 잠깐 본 그 기능입니다. 검색 **전에** 분류로 후보를
좁혀 엉뚱한 분류가 섞이는 걸 막습니다.

In [4]:
query = "관련 규정 알려줘"

print("───[ 필터 없음 ]───")
for d in db.similarity_search(query, k=3):
    print(" -", d.metadata["category"], ":", d.page_content)

print("\n───[ 필터: refund 분류만 ]───")
for d in db.similarity_search(query, k=3, filter={"category": "refund"}):
    print(" -", d.metadata["category"], ":", d.page_content)

───[ 필터 없음 ]───
 - refund : 환불 신청은 수령일 기준 7일 이내여야 합니다.
 - membership : 회원 등급은 누적 구매액으로 결정됩니다.
 - refund : 환불은 수령 후 7일 이내 신청 가능합니다.

───[ 필터: refund 분류만 ]───
 - refund : 환불 신청은 수령일 기준 7일 이내여야 합니다.
 - refund : 환불은 수령 후 7일 이내 신청 가능합니다.
 - refund : 환불 처리는 반품 확인 후 영업일 3일 이내입니다.


**이 셀이 한 일**: 필터를 거니 결과가 전부 `refund` 만
나옵니다.

> 💡 "원두 질문에 디저트 섞임" 같은 사고를 가장 확실하게 막는
> 처방입니다. 단, 분류 꼬리표가 미리 달려 있어야 합니다(2교시).

### 셀 4 — 처방② MMR: 다양하게 가져오기

기본 검색은 "가장 비슷한 것"만 골라, 비슷한 환불 문장이 많으면
거의 같은 청크만 가져옵니다. MMR은 다양성을 함께 봅니다.

In [ ]:
query = "환불 규정 알려줘"

print("───[ 기본 검색 (비슷한 것만) ]───")

print("\n───[ MMR 검색 (다양하게) ]───")


───[ 기본 검색 (비슷한 것만) ]───
 - 환불 처리는 반품 확인 후 영업일 3일 이내입니다.
 - 환불 신청은 수령일 기준 7일 이내여야 합니다.
 - 환불은 수령 후 7일 이내 신청 가능합니다.

───[ MMR 검색 (다양하게) ]───
 - 환불 처리는 반품 확인 후 영업일 3일 이내입니다.
 - 환불 신청은 수령일 기준 7일 이내여야 합니다.
 - 회원 등급은 누적 구매액으로 결정됩니다.


**이 셀이 한 일**: 기본 검색 vs MMR 검색을 비교했습니다.

> 💡 기본은 "7일 이내" 비슷한 문장이 몰릴 수 있고, MMR은
> "7일 신청 / 영업일 3일 처리"처럼 **서로 다른 측면**이 골고루
> 들어오는 경향이 있습니다. (`fetch_k`=후보를 넉넉히 본 뒤
> 그중 다양한 `k`개 선택)

### 셀 5 — 처방③ 질문 확장 (Multi-Query)

짧고 모호한 질문을, AI가 **여러 표현으로 바꿔 쓴 뒤** 각각
검색해 합칩니다. (AI를 더 쓰므로 OpenAI 키 필요)

In [ ]:
# langchain 버전에 상관없이 MultiQueryRetriever 를 안전하게 불러옵니다.
# (1.x = langchain_classic, 0.x = langchain)


print("MultiQueryRetriever import 성공 ✅")

MultiQueryRetriever import 성공 ✅


In [ ]:
if has_openai:
    from langchain_openai import ChatOpenAI
    # 위 셀에서 안전하게 import 된 MultiQueryRetriever 를 그대로 사용
    try:
        from langchain_classic.retrievers.multi_query import MultiQueryRetriever
    except ImportError:
        from langchain.retrievers.multi_query import MultiQueryRetriever

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    base = db.as_retriever(search_kwargs={"k": 2})

    # 기존 검색기를 한 겹 감싸기만 하면 됨
  

질문 확장으로 모은 청크 수: 3
 - 환불은 수령 후 7일 이내 신청 가능합니다.
 - 환불 신청은 수령일 기준 7일 이내여야 합니다.
 - 환불 처리는 반품 확인 후 영업일 3일 이내입니다.


**이 셀이 한 일**: 모호한 질문을 AI가 여러 각도로 다시
물어 더 폭넓게 청크를 모았습니다.

> 📌 정리: 세 처방의 사용법 —
> - **필터**: `similarity_search(q, filter={...})`
> - **MMR**: `as_retriever(search_type="mmr", ...)`
> - **질문확장**: `MultiQueryRetriever.from_llm(...)`

# 3️⃣ 단계 — 처방 조합 & 전후 비교

실무에선 처방을 섞어 쓰고, 항상 **전후를 비교**해 효과를
확인합니다.

### 셀 6 — 필터 + MMR 같이 쓰기

"원두 환불"처럼 분류도 좁히고 다양성도 원하면 둘을 함께 겁니다.

In [ ]:
query = "환불 관련 규정"

# search_type=mmr 와 filter 를 함께 지정


 - refund | 환불 처리는 반품 확인 후 영업일 3일 이내입니다.
 - refund | 환불 신청은 수령일 기준 7일 이내여야 합니다.
 - refund | 단순 변심 환불은 7일 안에만 받습니다.


**이 셀이 한 일**: 필터(분류 좁히기) + MMR(다양성)을 동시에
적용했습니다.

> 💡 결과가 전부 `refund` 이면서(필터) 서로 다른 측면(MMR)이면
> 성공. 조건 좁히기 + 다양성을 한 번에 얻는 실무 패턴입니다.

### 셀 7 — 전후 비교 (효과를 눈으로 확인)

처방의 효과는 **전후를 나란히 봐야** 압니다. 기본 vs 처방을
비교하는 함수를 만듭니다.

In [ ]:
def compare(query, retriever_before, retriever_after):
    print("질문:", query)
    print("\n[처방 전 — 기본 검색]")
    



compare("환불 규정 알려줘", basic, mmr)

질문: 환불 규정 알려줘

[처방 전 — 기본 검색]
  - 환불 처리는 반품 확인 후 영업일 3일 이내입니다.
  - 환불 신청은 수령일 기준 7일 이내여야 합니다.
  - 환불은 수령 후 7일 이내 신청 가능합니다.

[처방 후 — MMR]
  - 환불 처리는 반품 확인 후 영업일 3일 이내입니다.
  - 환불 신청은 수령일 기준 7일 이내여야 합니다.
  - 회원 등급은 누적 구매액으로 결정됩니다.


**이 셀이 한 일**: 처방 전후 검색 결과를 나란히 출력했습니다.

> 📌 정리: 처방은 **조합** 가능하고, 항상 **전후 비교**로 효과를
> 확인합니다. (이 습관이 7교시 '평가'의 출발점입니다)

# 4️⃣ 단계 — 위니브에서 검색 실패 고치기

진짜 위니브 데이터에서 1단계의 그 사고("원두 질문에 디저트")를
재현하고, 처방으로 고칩니다.

### 셀 8 — 위니브 벡터 DB 준비 

이미 배운 "불러오기 → 쪼개기 → 분류 → 벡터 DB"를 한 셀에
모았습니다.

In [10]:
import os
from collections import Counter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = {
    "01_faq.md":        "# 위니브 카페 자주 묻는 질문 (FAQ)\n\n스페셜티 커피 전문 쇼핑몰 위니브 카페 고객센터입니다.\n자주 묻는 질문을 모았습니다.\n\n---\n\n### Q. 주문은 어떻게 하나요?\nA. 원하시는 상품을 장바구니에 담은 뒤 결제 페이지에서 배송지와 결제수단을\n입력하시면 됩니다. 회원·비회원 모두 주문 가능합니다.\n\n### Q. 비회원도 주문할 수 있나요?\nA. 네, 비회원 주문이 가능합니다. 다만 비회원 주문은 적립금이 적립되지 않으며,\n회원 등급 혜택도 적용되지 않습니다.\n\n### Q. 주문을 취소하고 싶어요.\nA. 상품이 발송되기 전이라면 마이페이지 > 주문내역에서 직접 취소할 수 있습니다.\n이미 발송된 경우에는 반품 절차로 진행됩니다. 자세한 내용은 환불·반품 정책을\n확인해 주세요.\n\n### Q. 결제 수단은 무엇이 있나요?\nA. 신용·체크카드, 계좌이체, 간편결제(카카오페이·네이버페이)를 지원합니다.\n무통장입금은 현재 지원하지 않습니다.\n\n### Q. 선물 포장이 되나요?\nA. 네, 가능합니다. 주문 시 '선물포장' 옵션을 선택하시면 건당 2,000원에\n포장해 드립니다. 선물 메시지 카드도 함께 넣어드립니다.\n\n### Q. 주문한 상품은 언제 받을 수 있나요?\nA. 평일 오후 2시 이전 결제 건은 당일 발송되며, 보통 발송 후 1~2일 이내\n도착합니다. 지역에 따라 더 걸릴 수 있습니다. 배송 정책 문서를 참고해 주세요.\n\n### Q. 배송 조회는 어디서 하나요?\nA. 마이페이지 > 주문내역에서 송장번호와 배송 상태를 확인할 수 있습니다.\n발송 완료 시 알림 문자도 발송됩니다.\n\n### Q. 원두를 분쇄해서 보내주나요?\nA. 네, 주문 시 분쇄도(홀빈·드립·에스프레소 등)를 선택하실 수 있습니다.\n신선도를 위해 홀빈(분쇄 안 함) 구매를 권장드립니다.\n\n### Q. 재고가 없는 상품은 언제 다시 들어오나요?\nA. 품절 상품 페이지에서 '재입고 알림'을 신청하시면 입고 시 문자로\n안내드립니다. 원두는 로스팅 일정에 따라 보통 주 1회 입고됩니다.\n\n### Q. 영수증·세금계산서 발급이 되나요?\nA. 현금영수증은 결제 시 신청 가능하며, 세금계산서는 사업자 회원에 한해\n고객센터로 요청하시면 발급해 드립니다.\n\n### Q. 상품 문의는 어디로 하나요?\nA. 상품 상세페이지의 Q&A 게시판 또는 고객센터 채팅으로 문의해 주세요.\n운영 시간은 평일 오전 10시~오후 5시입니다.",
    "02_refund.md":     "# 위니브 카페 환불·반품·교환 정책\n\n본 정책은 전자상거래법을 따르며, 위니브 카페에서 구매한 모든 상품에 적용됩니다.\n\n## 1. 반품·환불 신청 기간\n\n- **단순 변심**: 상품 수령일로부터 7일 이내 신청 가능\n- **상품 하자·오배송**: 수령일로부터 14일 이내 신청 가능\n- 위 기간이 지나면 반품·환불이 제한될 수 있습니다.\n\n## 2. 환불 불가 품목 (중요)\n\n다음 품목은 단순 변심에 의한 반품·환불이 **불가**합니다.\n\n- **개봉한 원두 및 드립백**: 식품 위생상 개봉 시 재판매가 불가능합니다.\n  미개봉 상태에서는 7일 이내 반품 가능합니다.\n- **디저트류(케이크·쿠키·구움과자 등)**: 신선식품으로, 단순 변심 반품이\n  불가합니다. 단, 변질·파손 등 하자가 있는 경우 수령 14일 이내 교환·환불됩니다.\n- **주문 제작 굿즈(각인·커스텀 텀블러 등)**: 고객 요청으로 제작된 상품은\n  단순 변심 반품이 불가합니다.\n- 단, 위 품목도 **상품 하자나 오배송**인 경우에는 환불·교환이 가능합니다.\n\n## 3. 단순 변심 반품 시 비용\n\n- 단순 변심으로 반품할 경우 왕복 배송비(편도 3,000원 × 2 = 6,000원)는\n  고객이 부담합니다.\n- 상품 하자·오배송으로 인한 반품은 배송비를 위니브 카페가 전액 부담합니다.\n\n## 4. 환불 처리 절차 및 기간\n\n1. 마이페이지 > 주문내역에서 반품 신청\n2. 반품 상품 발송 (회수 택배 신청 또는 직접 발송)\n3. 반품 상품 도착 후 상태 확인\n4. 확인 완료 후 **영업일 기준 3일 이내** 환불 처리\n\n- 카드 결제는 카드사 정책에 따라 취소 반영까지 영업일 3~5일이 추가로\n  소요될 수 있습니다.\n- 적립금으로 결제한 부분은 적립금으로 환원됩니다.\n\n## 5. 교환\n\n- 동일 상품의 색상·옵션 변경은 재고가 있는 경우 교환 가능합니다.\n- 하자 상품의 교환 배송비는 위니브 카페가 부담합니다.\n- 단순 변심에 의한 교환 배송비는 고객이 부담합니다.\n- 디저트류는 교환이 어려워 환불로 처리될 수 있습니다.\n\n## 6. 주문 취소\n\n- 상품 발송 **전**에는 마이페이지에서 즉시 취소 가능하며 전액 환불됩니다.\n- 발송 **후**에는 취소가 불가하며, 수령 후 반품 절차로 진행됩니다.\n\n## 7. 환불이 제한되는 경우\n\n- 고객의 사용·훼손으로 상품 가치가 현저히 감소한 경우\n- 개봉 후 상태인 식품(원두·드립백·디저트)의 단순 변심\n- 반품 기간(7일/14일)을 경과한 경우\n- 위 사유라도 상품 자체 하자가 입증되면 환불 대상이 됩니다.",
    "03_shipping.md":   "# 위니브 카페 배송 정책\n\n## 1. 배송비\n\n| 구분 | 배송비 |\n|------|--------|\n| 기본 배송비 | 3,000원 |\n| 5만원 이상 구매 | 무료 |\n| 제주 지역 | 기본 배송비 + 3,000원 |\n| 도서산간 지역 | 기본 배송비 + 3,000원 |\n\n- 5만원 이상 구매 시 기본 배송비는 무료이나, 제주·도서산간 추가비\n  (3,000원)는 별도로 부과됩니다.\n\n## 2. 배송 기간\n\n- 평일 **오후 2시 이전** 결제 완료 건은 당일 발송됩니다.\n- 오후 2시 이후 결제 건은 다음 영업일에 발송됩니다.\n- 발송 후 도착까지 보통 1~2일 소요됩니다.\n- 제주·도서산간은 2~4일이 소요될 수 있습니다.\n- 주말·공휴일은 발송되지 않습니다.\n\n## 3. 원두 로스팅 일정\n\n- 원두는 신선도를 위해 **주문 후 로스팅(주문 로스팅)** 으로 발송됩니다.\n- 로스팅은 매주 화·금요일에 진행되며, 해당 요일 직전 주문이 함께\n  발송됩니다. 이로 인해 원두 단일 주문은 최대 3~4일이 소요될 수 있습니다.\n- 원두와 일반 상품을 함께 주문하면 로스팅 일정에 맞춰 합포장 발송됩니다.\n\n## 4. 배송 조회\n\n- 마이페이지 > 주문내역에서 송장번호로 실시간 조회가 가능합니다.\n- 발송 완료 시 등록된 휴대폰으로 알림 문자가 발송됩니다.\n\n## 5. 배송 관련 안내\n\n- 택배사는 일반 택배(CJ대한통운)를 이용합니다.\n- **새벽배송·당일배송 서비스는 운영하지 않습니다.**\n- 부재 시 택배사 정책에 따라 경비실 위탁 또는 재배송됩니다.\n- 배송지 오기입으로 인한 반송 시 재발송 배송비는 고객이 부담합니다.\n\n## 6. 해외 배송\n\n- 현재 해외 배송은 지원하지 않습니다. 국내 배송만 가능합니다.",
    "04_catalog.md":    "# 위니브 카페 상품 정보 카탈로그\n\n각 상품에는 `카테고리` 정보가 있습니다. (메타데이터 필터 실습에 사용)\n카테고리: 원두 / 드립백 / 디저트 / 홈카페기기 / 바리스타굿즈\n\n---\n\n## [원두] 에티오피아 예가체프 G1\n\n- 카테고리: 원두\n- 가격: 18,000원 (250g)\n- 설명: 자스민·베르가못 향이 도는 밝은 산미의 워시드 원두입니다.\n- 보관: 직사광선을 피해 서늘한 곳에 보관, 개봉 후 2주 이내 소비 권장\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [원두] 브라질 산토스 NY2\n\n- 카테고리: 원두\n- 가격: 14,000원 (250g)\n- 설명: 견과류·다크초콜릿의 묵직한 바디. 에스프레소 베이스로 인기.\n- 보관: 밀폐 용기 보관, 개봉 후 2주 이내 소비 권장\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [드립백] 데일리 블렌드 드립백 10개입\n\n- 카테고리: 드립백\n- 가격: 9,000원 (10개입)\n- 설명: 사무실·여행용 휴대 간편 드립백. 균형 잡힌 블렌드.\n- 보관: 개별 포장, 실온 보관\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [디저트] 클래식 파운드 케이크\n\n- 카테고리: 디저트\n- 가격: 12,000원\n- 설명: 버터 풍미가 진한 수제 파운드 케이크. 커피와 페어링 추천.\n- 보관: 냉장 보관, 수령 후 3일 이내 섭취 권장\n- 환불: 신선식품으로 단순 변심 반품 불가 / 변질·파손 시 14일 이내 교환·환불\n\n## [디저트] 비스코티 세트 (6개)\n\n- 카테고리: 디저트\n- 가격: 9,500원\n- 설명: 아몬드·헤이즐넛 두 가지 맛 구움과자 세트.\n- 보관: 실온 보관, 수령 후 7일 이내 섭취 권장\n- 환불: 신선식품으로 단순 변심 반품 불가 / 하자 시 14일 이내 교환·환불\n\n## [홈카페기기] 핸드드립 세트 (드리퍼+서버)\n\n- 카테고리: 홈카페기기\n- 가격: 38,000원\n- 설명: 입문용 드리퍼와 600ml 유리 서버 구성.\n- 보관: -\n- 환불: 미사용 7일 이내 가능 / 하자 시 14일 이내 교환\n\n## [홈카페기기] 전동 그라인더 GR-200\n\n- 카테고리: 홈카페기기\n- 가격: 89,000원\n- 설명: 40단계 분쇄도 조절 코니컬 버 그라인더.\n- 보관: -\n- 환불: 미사용 7일 이내 가능 / 하자(작동 불량) 시 14일 이내 무상 교환\n\n## [바리스타굿즈] 위니브 로고 머그컵\n\n- 카테고리: 바리스타굿즈\n- 가격: 12,000원\n- 설명: 300ml 도자기 머그. 위니브 카페 로고 인쇄.\n- 보관: -\n- 환불: 미사용 7일 이내 가능\n\n## [바리스타굿즈] 커스텀 각인 텀블러 (주문제작)\n\n- 카테고리: 바리스타굿즈\n- 가격: 28,000원\n- 설명: 원하는 문구를 각인해 제작하는 보온 텀블러.\n- 보관: -\n- 환불: **주문 제작 상품으로 단순 변심 반품 불가** / 하자 시 교환 가능",
    "05_membership.md": "# 위니브 카페 멤버십·적립 안내\n\n## 1. 회원 등급\n\n회원 등급은 **최근 6개월 누적 구매액**을 기준으로 매월 1일 자동 산정됩니다.\n\n| 등급 | 최근 6개월 누적 구매액 | 적립률 |\n|------|----------------------|--------|\n| 베이직 | 0원 ~ | 1% |\n| 실버 | 10만원 이상 | 2% |\n| 골드 | 30만원 이상 | 3% |\n| VIP | 60만원 이상 | 5% |\n\n- 등급은 매월 1일 재산정되며, 기준 미달 시 하향될 수 있습니다.\n- 신규 가입 시 기본 등급은 베이직입니다.\n\n## 2. 적립금\n\n- 구매 확정 시 등급별 적립률에 따라 적립됩니다.\n- 적립금은 **적립일로부터 1년** 경과 시 자동 소멸됩니다.\n- 적립금은 1,000원 이상부터 사용 가능하며, 1원 단위로 사용할 수 있습니다.\n- 비회원 주문은 적립이 적용되지 않습니다.\n\n## 3. 신규 가입 혜택\n\n- 신규 회원가입 시 3,000원 웰컴 적립금이 즉시 지급됩니다.\n- 웰컴 적립금의 유효기간은 지급일로부터 30일입니다.\n\n## 4. 생일 쿠폰\n\n- 생일이 등록된 회원에게 생일月 1일에 10% 할인 쿠폰이 발급됩니다.\n- 생일 쿠폰은 발급일로부터 그 달 말일까지 사용 가능합니다.\n- 일부 할인 상품에는 중복 적용되지 않습니다.\n\n## 5. 적립금 환불 처리\n\n- 적립금으로 결제한 주문을 환불할 경우, 사용한 적립금은 적립금으로\n  환원됩니다(현금 환불 아님).\n- 환원된 적립금의 유효기간은 원래 적립금의 잔여 기간을 따릅니다.\n\n## 6. 등급 혜택 요약\n\n- 골드 등급 이상: 무료 선물포장 제공\n- VIP 등급: 신상품 우선 구매 및 분기별 시음 원두 증정",
}
os.makedirs("winib_docs", exist_ok=True)
for filename, content in documents.items():
    with open(f"winib_docs/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

all_docs = DirectoryLoader("winib_docs", glob="*.md",
                           loader_cls=TextLoader,
                           loader_kwargs={"encoding": "utf-8"}).load()
chunks = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=30).split_documents(all_docs)

category_map = {"01_faq.md": "faq", "02_refund.md": "refund",
                "03_shipping.md": "shipping", "04_catalog.md": "catalog",
                "05_membership.md": "membership"}
for c in chunks:
    # source 가 없으면 빈 문자열로 (방어), 그 뒤 category 부여
    src_name = os.path.basename(c.metadata.get("source", ""))
    c.metadata["category"] = category_map.get(src_name, "기타")

winib_db = Chroma.from_documents(chunks, embeddings)
print("위니브 벡터 DB 준비 완료 ✅")
print("분류별 청크:", dict(Counter(c.metadata["category"] for c in chunks)))

위니브 벡터 DB 준비 완료 ✅
분류별 청크: {'refund': 5, 'faq': 5, 'catalog': 7, 'shipping': 4, 'membership': 4}


**이 셀이 한 일**: 위니브 문서 5개를 분류 꼬리표 달아
벡터 DB에 저장했습니다. (2~3교시 그대로)

### 셀 9 — 검색 실패 재현 (원두 질문에 디저트 섞임)

 "원두 환불"은 "디저트 환불"과
의미가 비슷해서, 기본 검색이 디저트/상품 청크를 섞어 옵니다.

In [ ]:
query = "원두 환불 규정 알려줘"



[기본 검색] 가져온 청크의 분류:
  - refund     | 환불 처리는 반품 확인 후 영업일 3일 이내입니다.
  - refund     | - **개봉한 원두 및 드립백**: 식품 위생상 개봉 시 재판매가 불가능
  - catalog    | ## [원두] 브라질 산토스 NY2

- 카테고리: 원두
- 가격: 14
  - refund     | ## 6. 주문 취소

- 상품 발송 **전**에는 마이페이지에서 즉시


**이 셀이 한 일**: 기본 검색으로 "원두 환불"을 검색했습니다.

> 💡 `catalog`(상품) 등 원두 환불과 무관한 분류가 섞여 들어오면,
> 그게 바로 1단계에서 본 "엉뚱한 분류 섞임" 증상입니다.

### 셀 10 — 처방① 필터로 고치기

"환불 규정" 질문은 `refund` 청크만 보면 됩니다. 필터로
디저트 청크를 원천 차단합니다.

In [ ]:


print("[필터 적용] 가져온 청크의 분류:")


[필터 적용] 가져온 청크의 분류:
  - refund     | 환불 처리는 반품 확인 후 영업일 3일 이내입니다.
  - refund     | - **개봉한 원두 및 드립백**: 식품 위생상 개봉 시 재판매가 불가능
  - refund     | ## 6. 주문 취소

- 상품 발송 **전**에는 마이페이지에서 즉시
  - refund     | - 카드 결제는 카드사 정책에 따라 취소 반영까지 영업일 3~5일이 추가


**이 셀이 한 일**: 필터를 거니 전부 `refund` 만 나옵니다.

> 💡 **검색 부품(retriever)만 바꿨을 뿐, 4교시 챗봇 나머지는
> 그대로**입니다. 검색만 갈아 끼우면 되는 게 RAG의 장점입니다.

### 셀 11 — 고친 검색으로 RAG 답변까지 (전후 비교)

기본 검색 챗봇 vs 필터 검색 챗봇의 답을 비교합니다.
(OpenAI 키가 있을 때 — 4교시 챗봇 구조 그대로 검색만 교체)

In [13]:
if has_openai:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.runnables import RunnablePassthrough

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "위니브 카페 상담원입니다. [참고자료]만 보고 답하고, "
                   "출처를 (출처: 파일명) 으로 적으세요.\n\n[참고자료]\n{context}"),
        ("human", "{question}"),
    ])

    def fmt(docs):
        # metadata 에 source 가 없을 수도 있으므로 .get() 으로 안전하게 꺼냄
        blocks = []
        for d in docs:
            source = d.metadata.get("source", "문서")   # 없으면 "문서"
            filename = os.path.basename(source)
            blocks.append(f"[출처: {filename}]\n{d.page_content}")
        return "\n\n".join(blocks)

    # 검색기만 바꿔 끼우는 챗봇 만들기 함수
    def make_chatbot(retriever):
        return ({"context": retriever | fmt, "question": RunnablePassthrough()}
                | prompt | llm | StrOutputParser())

    q = "원두 환불 규정 알려줘"
    print("[기본 검색 챗봇]\n", make_chatbot(basic_retriever).invoke(q))
    print("\n[필터 검색 챗봇]\n", make_chatbot(refund_retriever).invoke(q))
else:
    print("OpenAI 키가 없어 답변 생성은 건너뜁니다 (셀 9·10 검색 차이로 효과는 확인됨).")

[기본 검색 챗봇]
 원두의 환불 규정은 다음과 같습니다:

- 미개봉 상태에서는 7일 이내에 반품이 가능합니다.
- 개봉한 원두는 식품 위생상 재판매가 불가능하므로 환불이 불가합니다. 

(출처: 02_refund.md)

[필터 검색 챗봇]
 원두의 환불 규정은 다음과 같습니다:

- 개봉한 원두는 식품 위생상 재판매가 불가능하므로, 개봉 후에는 환불이 불가합니다.
- 미개봉 상태의 원두는 7일 이내에 반품이 가능합니다.
- 상품 하자나 오배송인 경우에는 환불이 가능합니다.

(출처: 02_refund.md)


**이 셀이 한 일**: 검색만 다른 두 챗봇의 답을 비교했습니다.

> 💡 필터 챗봇 답이 디저트 얘기 없이 원두/환불에 집중되면 성공.
> **같은 챗봇, 검색만 교체**했을 뿐인데 답 품질이 달라집니다.

---
# 🎓 배운 것 정리

| 단계 | 한 일 | 핵심 |
|------|-------|------|
| 1️⃣ 이론 | RAG 틀리면 검색을 의심 | 증상별 3처방 |
| 2️⃣ 기본 | 필터 / MMR / 질문확장 | 검색 부품만 교체 |
| 3️⃣ 응용 | 처방 조합 + 전후 비교 | 7교시 평가의 기초 |
| 4️⃣ 실습 | 위니브 실패 재현 → 필터 교정 | 같은 챗봇, 검색만 개선 |

**한 문장 요약**: RAG가 틀리면 AI가 아니라 **검색**을 먼저
의심하고, 증상에 맞는 처방(필터·MMR·질문확장)으로 검색만 고친다.

**다음 시간**: 지금 챗봇은 질문을 하나씩 따로 받습니다.
"그거 배송은?" 같은 **대화 맥락을 기억하는** 챗봇으로 키웁니다.

---
## 🤔 생각해보기 (정답 없음 — 토론용)

검색을 많이 개선했습니다. 그런데 실제 상담은 한 번 묻고
끝나지 않습니다.

```
🧑 "에티오피아 예가체프 원두 환불돼요?"
🤖 "미개봉 7일 이내 가능합니다. (출처: 02_refund.md)"
🧑 "그럼 그거 배송은 며칠 걸려요?"
🤖 "어떤 상품을 말씀하시는지 알려주시겠어요?" 😶
```

1. "그거 배송은?" 에서 챗봇이 막힌 이유는? (챗봇은 방금
   대화를 기억할까요?)
2. 검색기에 "그거 배송은?" 만 넘기면 "그거"가 뭔지 모릅니다.
   검색 전에 무엇이 필요할까요?
3. 멀티턴 대화를 기억하게 하려면 어떤 정보를 어디에 저장하고
   어떻게 활용할까요? 자유롭게 적어보세요.

> ✍️ 메모:
> - 1번:
> - 2번:
> - 3번:
